In [55]:
import os
import pandas as pd
from typing import List, Tuple
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import json
import torch

font = {'size': 16}

matplotlib.rc('font', **font)

In [56]:
DATASET_PATH = "../dataset"
INDEX_FIELD = "timestamp"
DATA_FIELD = "num_request"
CPD_CANDIDATE_ROOT = "../change_point_detection/offline_detection/cpd_candidate"
OCPD_RESULT_ROOT = "../ocpd/result"
N_LOOKBACK = 4
N_PREDICT = 2


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [57]:
def get_data_file_list(dataset_path: str) -> List[str]:
    return os.listdir(dataset_path)

In [58]:
def read_dataset(csv_path: str,index_field:str,data_field:str) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(csv_path)
    return df[index_field].to_numpy(), df[data_field].to_numpy()

In [59]:
def read_candidate_cpds(path: str) -> List[int]:
    candidate_cpds = None
    with open(path, "r") as f:
        candidate_cpds = json.load(f)
    return candidate_cpds

In [60]:

class Simulator:
    def __init__(self, workload: np.ndarray, capacity_per_pod: float, cpd_idx: List, sla: float, fluctuation: float) -> None:
        self.t = 0
        self.sla = sla
        self.upper_sla = sla+fluctuation
        self.lower_sla = sla-fluctuation
        self.workload = workload
        self.capaciy_per_pod = capacity_per_pod
        self.cpd_idx = cpd_idx
        self.r_cpu_list = []
        self.n_pod_list = []
        self.n_scale = 0
        self.workload_list = []

    def run(self):
        n_pod = int(np.ceil(self.workload[0]/(self.sla*self.capaciy_per_pod)))
        while self.t < len(self.workload)-2:
            current_workload = self.workload[self.t]
            self.workload_list.append(current_workload)
            r_cpu = 1.0*current_workload/(n_pod*self.capaciy_per_pod)
            self.r_cpu_list.append(r_cpu)
            self.n_pod_list.append(n_pod)
            if self.t in self.cpd_idx:
                future_workload = np.average([self.workload[self.t+1], self.workload[self.t+2]])
                target_n_pod = int(np.ceil(future_workload/(self.sla*self.capaciy_per_pod)))
                if n_pod != target_n_pod:
                    n_pod = target_n_pod
                    self.n_scale += 1
            self.t += 1
    
    def r_sla_violate(self):
        return 1.0*(len(np.where(np.array(self.r_cpu_list) > self.upper_sla)[0])+len(np.where(np.array(self.r_cpu_list) < self.lower_sla)[0]))/len(self.r_cpu_list)

In [61]:
workload_to_skip_list = ["workload_1998-06-13", "workload_1998-06-14", "workload_1998-06-20", "workload_1998-06-21", "workload_1998-06-27", "workload_1998-06-28","workload_1998-07-04"]

In [62]:
def dump_dir(dir: str, path: str):
    with open(path, "w") as f:
        json.dump(dir, f, indent=4)

In [63]:
n_scale_dict = {}
r_cpu_dict = {}
n_pod_dict = {}
workload_dict = {}
r_sla_vio_dict={}
data_file_list = get_data_file_list(DATASET_PATH)
for file_name in data_file_list:
    workload_name = file_name.split(".")[0]
    if workload_name in workload_to_skip_list:
        continue
    print("read %s" % (file_name))
    np_index, np_data = read_dataset(os.path.join(DATASET_PATH, file_name), INDEX_FIELD, DATA_FIELD)
    np_data=np.floor(np_data/10.0)
    ocpd_result=None
    with open(os.path.join(OCPD_RESULT_ROOT,workload_name+".json"),"r") as f:
        ocpd_result=json.load(f)
    simulator=Simulator(np_data,2500,ocpd_result,0.5,0.1)
    simulator.run()
    n_scale_dict[workload_name] = simulator.n_scale
    r_cpu_dict[workload_name] = simulator.r_cpu_list
    n_pod_dict[workload_name] = simulator.n_pod_list
    workload_dict[workload_name] = simulator.workload_list
    r_sla_vio_dict[workload_name]=simulator.r_sla_violate()
dump_dir(n_scale_dict,"ocpdas_result/n_scale.json")
dump_dir(r_cpu_dict,"ocpdas_result/r_cpu.json")
dump_dir(n_pod_dict,"ocpdas_result/n_pod.json")
dump_dir(workload_dict,"ocpdas_result/workload.json")
dump_dir(r_sla_vio_dict,"ocpdas_result/r_sla_violate.json")

read workload_1998-06-10.csv


KeyboardInterrupt: 